# Alaska salmon data cleaning

This notebook holds the code to clean the Alaskan salmon dataset found on [KNB](https://knb.ecoinformatics.org/view/doi:10.5063/F1707ZTM). This data set spans 1922 to 2017 and contains salmon measurements, sample locations, and type of salmon capture. The final result is a local relational database that makes this data more usable.

In [1]:
# import in libraries
import pandas as pd
import numpy as np
import os

In [ ]:
# load in data
salmon_data = pd.read_csv("Alaskan-salmon1922-2017/data/ASL_master.csv")
length_type = pd.read_csv("Alaskan-salmon1922-2017/data/length_type_lookup.csv")
location = pd.read_csv("Alaskan-salmon1922-2017/data/Locations_subdistricts_uniqueID.csv")
project_type = pd.read_csv("Alaskan-salmon1922-2017/data/ASLProjectType.csv")

/var/folders/kj/1ybgv25d7zd06v9ndqdccd2r0000gn/T/ipykernel_38349/1601985448.py:2: DtypeWarning: Columns (1,3,8,11,12,13,15,16,17,18,26) have mixed types. Specify dtype option on import or set low_memory=False.
  salmon_data = pd.read_csv("Alaskan-salmon1922-2017/data/ASL_master.csv")


Look at the datatypes of the data.

In [3]:
salmon_data.dtypes

Species                      object
Length.Measurement.Type      object
sampleYear                  float64
ASLProjectType               object
LocationID                   object
sampleDate                   object
Length                      float64
Weight                      float64
Sex                          object
Salt.Water.Age              float64
DataSource                   object
cardNo                       object
fishNum                      object
Age.Error                    object
Fresh.Water.Age             float64
Sex.Determination.Method     object
subSystem                    object
Flag                         object
Gear                         object
SASAP.Region                 object
LocationUnique               object
DistrictID                  float64
Sub.DistrictID              float64
Stat.area                   float64
Lat                         float64
Lon                         float64
AWC_CODE                     object
dtype: object

### The first step is to make a primary key for our main dataframe
Add an ID column. 

In [5]:
# add an id column to the salmon dataframe
salmon_data['id'] = salmon_data.index

In [46]:
# initial look at the dataframe
salmon_data.head()

,Species,Length.Measurement.Type,sampleYear,ASLProjectType,LocationID,sampleDate,Length,Weight,Sex,Salt.Water.Age,Fresh.Water.Age,Gear,Stat.area,DataSource
0,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward
1,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,0.0,troll,10510.0,ADFG Southeast and Westward
2,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,1.0,troll,10510.0,ADFG Southeast and Westward
3,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,3.0,0.0,troll,10510.0,ADFG Southeast and Westward
4,chinook,length not taken,1992.0,commercial catch,Affleck Canal/Spanish Is/Louise Cove,1992-03-31,NaN,NaN,examined but did not identify,4.0,0.0,troll,10510.0,ADFG Southeast and Westward


Check the new datatypes.

In [6]:
salmon_data.dtypes

Species                      object
Length.Measurement.Type      object
sampleYear                  float64
ASLProjectType               object
LocationID                   object
sampleDate                   object
Length                      float64
Weight                      float64
Sex                          object
Salt.Water.Age              float64
DataSource                   object
cardNo                       object
fishNum                      object
Age.Error                    object
Fresh.Water.Age             float64
Sex.Determination.Method     object
subSystem                    object
Flag                         object
Gear                         object
SASAP.Region                 object
LocationUnique               object
DistrictID                  float64
Sub.DistrictID              float64
Stat.area                   float64
Lat                         float64
Lon                         float64
AWC_CODE                     object
id                          

Yay! It looks like all of the data is in the right data type except for the sampleYear and sampleDate. 
 

In [ ]:
# convert salmon year to an int
salmon_data['sampleYear'] = salmon_data['sampleYear'].astype('Int64')

In [ ]:
# 

We don't need all of the columns for this analysis. 

In [19]:
salmon_data = salmon_data[['Species', 'Length.Measurement.Type', 'sampleYear', 'ASLProjectType', 'LocationID', 'sampleDate', 'Length', 'Weight', 'Sex','Salt.Water.Age', 'Fresh.Water.Age', 'Gear', 'Stat.area', 'DataSource']]

salmon_data

# Length dataset cleaning 

In [21]:
length_type.head()

,Length.Measurement.Type,Length_fill,LocationID,Species,ASLProjectType,sampleYear
0,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1998.0
1,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,1999.0
2,mid-eye to fork of tail,NaN,108 Creek,coho,escapement,2000.0
3,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2003.0
4,mid-eye to fork of tail,NaN,18 Mile Slough,chum,escapement,2004.0


In [27]:
length_type.dtypes

Length.Measurement.Type     object
Length_fill                 object
LocationID                  object
Species                     object
ASLProjectType              object
sampleYear                 float64
dtype: object

In [ ]:
# check all of the unique values of the year column
length_type['sampleYear'].unique()

array([1998., 1999., 2000., 2003., 2004., 1984., 1983., 1985., 1986.,
       1987., 1988., 1990., 1991., 1992., 1993., 2002., 2005., 2006.,
       2007., 2008., 1989., 1996., 2010., 2011., 2013., 2014., 1994.,
       1995., 1997., 2001., 2009., 2012., 2015., 1982., 1965., 1967.,
       1964., 1970., 1971., 1972., 1974., 1975., 1963., 1966., 1968.,
       1969., 1973.,   nan, 1976., 1960., 1978., 2016., 1979., 1980.,
       1981., 1977., 1961., 2017., 1962., 1959., 1958., 1957.])

In [35]:
# change year from float ot int
length_type['sampleYear'] = length_type['sampleYear'].astype('Int64')

In [39]:
location.dtypes

SASAP.Region               object
SASAP.Region_Corrected     object
Location                   object
ASLProjectType             object
District                   object
Sub.District               object
LocationUnique             object
DistrictID                float64
Sub.DistrictID            float64
Stat.area                 float64
Lat                       float64
Lon                       float64
AWC_CODE                   object
source                     object
LocationID                 object
dtype: object

In [40]:
location.head()

,SASAP.Region,SASAP.Region_Corrected,Location,ASLProjectType,District,Sub.District,LocationUnique,DistrictID,Sub.DistrictID,Stat.area,Lat,Lon,AWC_CODE,source,LocationID
0,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,NaN,commercial catch,302,19,Aleutians-commercial catch-302,302.0,NaN,NaN,NaN,NaN,NaN,NaN,Aleutians
1,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,commercial catch,315,11,Bear River-commercial catch-31511,315.0,11.0,31511.0,NaN,NaN,NaN,NaN,Bear River
2,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,escapement,315,11,Bear River-escapement-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River
3,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,315,11,Bear River-test fishing-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River
4,Alaska Peninsula and Aleutian Islands,Alaska Peninsula and Aleutian Islands,Bear River,test fishing,316,11,Bear River-test fishing-31511,315.0,11.0,31511.0,56.0389,-160.2734,315-11-10200,ADFG,Bear River


In [ ]:
# subset for only necessary rows
location = location[['LocationUnique', 'SASAP.Region_Corrected','Location', 'ASLProjectType', 'District', 'Sub.District', 'Lat', 'Lon']]

In [44]:
gear.head()

,Gear,SASAP.Gear
0,NaN,NaN
1,handpicked or carcass,handpicked or carcass
2,beach seine,seine
3,sport hook and line,sport hook and line
4,weir,weir
